<a href="https://colab.research.google.com/github/mahammadaftab/GenAI-Lab/blob/main/Experiment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install spacy transformers
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 33.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import spacy
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [3]:
nlp = spacy.load('en_core_web_md')

In [4]:
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def get_similar_words(word, top_n=5):
    token = nlp(word)
    similar_words = []

    for vocab_word in nlp.vocab:
        if vocab_word.has_vector and vocab_word.is_lower:
            similarity = token.similarity(vocab_word)
            if similarity > 0.5:
                similar_words.append((vocab_word.text, similarity))

    similar_words = sorted(similar_words, key=lambda x: x[1], reverse=True)[:top_n]
    return [word[0] for word in similar_words]


In [6]:
def generate_response(prompt):
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    outputs = model.generate(inputs, max_length=100, num_return_sequences=1, no_repeat_ngram_size=2, top_k=50)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response

In [7]:
original_prompt = "Describe a beautiful forest scene."

In [8]:
similar_beautiful = get_similar_words("beautiful")
similar_forest = get_similar_words("forest")
similar_scene = get_similar_words("scene")

In [9]:
enriched_prompt = f"Describe a {'/'.join(similar_beautiful)} {'/'.join(similar_forest)} {'/'.join(similar_scene)}."

In [10]:
original_response = generate_response(original_prompt)
enriched_response = generate_response(enriched_prompt)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [11]:
print("Original Prompt Response:")
print(original_response)
print("\nEnriched Prompt Response:")
print(enriched_response)

Original Prompt Response:
Describe a beautiful forest scene.

The forest is a very beautiful place. It is very quiet. The trees are very tall. There are many trees. You can see the trees in the distance. They are not very big. But they are small. And they have a lot of leaves. So you can imagine the forest. I think it is beautiful. We have to go to the other side of the mountain. If we go there, we will see a forest that is not so

Enriched Prompt Response:
Describe a is/these/a/beautiful/am forest scene/nuthin/somethin/doin/havin.

The scene is a beautiful forest. The scene has a lot of trees. It's a forest that's very beautiful. I think it's the best forest in the world. And I'm not saying it is the most beautiful, but it has the highest quality of life. So I don't think that it should be called a "beaut
